In [21]:
#read from data frame
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
df_nyctaxi=spark.read.table("samples.nyctaxi.trips")
display(df_nyctaxi)

AnalysisException: [REQUIRES_SINGLE_PART_NAMESPACE] spark_catalog requires a single-part namespace, but got `samples`.`nyctaxi`. SQLSTATE: 42K05

In [ ]:
# Verify Databricks Connection using .venv_dbx on top right kernel selection
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

df = spark.range(5)

df.show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [ ]:
# verify using local pyspark (.ven_local_pyspark)
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("LocalSparkTest")
    .getOrCreate()
)

df = spark.createDataFrame(
    [("Dev", 100), ("Sam", 200)],
    ["name", "score"]
)

df.show()

26/05/07 09:18:50 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


+----+-----+
|name|score|
+----+-----+
| Dev|  100|
| Sam|  200|
+----+-----+



In [ ]:
# works with .venv_dvx kernel but not with .venv_local_pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

df = spark.read.table("samples.nyctaxi.trips")

df.show(5)

+--------------------+---------------------+-------------+-----------+----------+-----------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|fare_amount|pickup_zip|dropoff_zip|
+--------------------+---------------------+-------------+-----------+----------+-----------+
| 2016-02-13 21:47:53|  2016-02-13 21:57:15|          1.4|        8.0|     10103|      10110|
| 2016-02-13 18:29:09|  2016-02-13 18:37:23|         1.31|        7.5|     10023|      10023|
| 2016-02-06 19:40:58|  2016-02-06 19:52:32|          1.8|        9.5|     10001|      10018|
| 2016-02-12 19:06:43|  2016-02-12 19:20:54|          2.3|       11.5|     10044|      10111|
| 2016-02-23 10:27:56|  2016-02-23 10:58:33|          2.6|       18.5|     10199|      10022|
+--------------------+---------------------+-------------+-----------+----------+-----------+
only showing top 5 rows


In [ ]:
#download dataset for local to run this locally

from pyspark.sql import SparkSession
spark=SparkSession.builder.getOrCreate()
df = spark.read.table("samples.nyctaxi.trips")
# df.count() --21932
pdf = df.limit(22000).toPandas()
pdf.to_csv("nyctaxi_sample.csv", index=False)


21932

In [22]:
# now read locally from csv file
from pyspark.sql import SparkSession
spark=SparkSession.builder.getOrCreate()
df_nyc_local=spark.read.csv("basics/nyctaxi_sample.csv",header=True,inferSchema=True)
# df_nyc_local.show(10)

# df_nyc_local.printSchema()


In [23]:
# another way
# df_nyc_local = (
#     spark.read
#     .format("csv")
#     .option("header", "true")
#     .option("inferSchema", "true")
#     .load("basics/nyctaxi_sample.csv")
# )

# df_nyc_local.show(5)

In [ ]:
from pyspark.sql.functions import col,when,round
# df_nyc_local.show(10)
# df_nyc_local.printSchema()
df_new=df_nyc_local.withColumn("cust_category", when(col("fare_amount")>100, "Premium").when(col("fare_amount")>50,"Standrad").otherwise("Customer"))
# df_new.filter(col("fare_amount")>50).show() --Recommended
# df_new.filter("fare_amount > 50").show() ---SQL-style string syntax

+--------------------+---------------------+-------------+-----------+----------+-----------+-------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|fare_amount|pickup_zip|dropoff_zip|cust_category|
+--------------------+---------------------+-------------+-----------+----------+-----------+-------------+
| 2016-02-03 05:16:34|  2016-02-03 05:35:48|         15.7|       52.0|     10009|      11422|     Standrad|
| 2016-02-20 19:28:12|  2016-02-20 20:08:33|         19.6|       54.5|     11422|      11238|     Standrad|
| 2016-02-15 16:41:54|  2016-02-15 17:38:20|        18.43|       52.0|     11422|      10011|     Standrad|
| 2016-02-28 13:48:59|  2016-02-28 14:19:36|         15.5|       52.0|     10017|      11422|     Standrad|
| 2016-02-24 07:15:56|  2016-02-24 07:57:07|        17.28|       52.0|     10167|      11422|     Standrad|
| 2016-02-15 11:10:38|  2016-02-15 11:41:07|         18.0|       52.0|     10020|      11422|     Standrad|
| 2016-02-22 10:26:26|  2016

In [45]:
#Group by Agg and joins
from pyspark.sql.functions import col,sum,avg,count
# df_nyc_local.show()
df_summary = (
    df_nyc_local
    .groupBy("pickup_zip")
    .agg(
        sum(col("fare_amount")).alias("total_fare"),
        round(avg(col("fare_amount")), 2).alias("avg_fare"),
        count('*').alias('trips')
    )
    .filter(col("trips")>100)
    .orderBy(col('avg_fare').desc())
)
df_summary.show()

+----------+----------+--------+-----+
|pickup_zip|total_fare|avg_fare|trips|
+----------+----------+--------+-----+
|     11422|   19209.5|   44.78|  429|
|     11371|  14798.84|   30.64|  483|
|     10280|    1681.5|   16.33|  103|
|     10005|    1602.5|   15.87|  101|
|     10271|    1974.5|    15.8|  125|
|     10006|    1902.0|   15.22|  125|
|     10282|    2357.0|   14.46|  163|
|     10007|    2329.3|   13.86|  168|
|     10013|    3777.5|   13.84|  273|
|     10278|    2169.5|   12.99|  167|
|     10103|   7351.01|   12.61|  583|
|     10002|    7494.0|   12.23|  613|
|     10171|    2372.5|   12.04|  197|
|     10154|    2903.0|    11.9|  244|
|     10020|    5417.5|    11.7|  463|
|     10025|    3153.0|   11.68|  270|
|     10167|    3347.0|   11.58|  289|
|     10018|  11541.51|    11.4| 1012|
|     10012|    9467.0|   11.35|  834|
|     10017|    7847.0|   11.31|  694|
+----------+----------+--------+-----+
only showing top 20 rows


In [46]:
#joins
#lets create dataframe
zip_info=spark.createDataFrame(
    [
        (10001,'Chelsea','Manhattan'),
        (10002,'lower Manhattan','Manhattan')
    ]
)
zip_info.show()

+-----+---------------+---------+
|   _1|             _2|       _3|
+-----+---------------+---------+
|10001|        Chelsea|Manhattan|
|10002|lower Manhattan|Manhattan|
+-----+---------------+---------+



In [ ]:
df_summary = (
    df_nyc_local
    .filter(col("fare_amount") > 0)
    .groupBy("pickup_zip")
    .agg(
        sum(col("fare_amount")).alias("total_fare"),
        round(avg(col("fare_amount")), 2).alias("avg_fare"),
        count("*").alias("trips")
    )
)
df_summary.show()

+----------+----------+--------+-----+
|pickup_zip|total_fare|avg_fare|trips|
+----------+----------+--------+-----+
|     10468|       8.0|     4.0|    2|
|     10032|     249.5|   16.63|   15|
|     10013|    3777.5|   13.84|  273|
|     10022|    5106.5|    9.84|  519|
|     10162|    4172.5|   10.08|  414|
|     10018|  11541.51|   11.42| 1011|
|     11106|     399.5|   10.24|   39|
|     11237|     186.0|    12.4|   15|
|     10011|   12321.5|   10.91| 1129|
|     11103|     176.5|   11.03|   16|
|     11422|   19209.5|   44.78|  429|
|     11201|    1077.0|   13.63|   79|
|     10020|    5417.5|    11.7|  463|
|      7311|      60.0|    60.0|    1|
|      7718|       5.0|     5.0|    1|
|     11368|      52.0|    52.0|    1|
|     11423|      11.5|    11.5|    1|
|     10006|    1902.0|   15.22|  125|
|     10037|     348.5|    13.4|   26|
|     11226|      75.0|   18.75|    4|
+----------+----------+--------+-----+
only showing top 20 rows


26/05/08 10:31:48 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 939574 ms exceeds timeout 120000 ms
26/05/08 10:31:48 WARN SparkContext: Killing executors is not supported by current scheduler.
26/05/08 10:32:34 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:674)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1363)
	at o

In [8]:
# running using .venv_dvx
# spark.sql("Select 5+3" ).show() --8
# Create a table
# spark.sql("Create table if not exists default.emp(eid int, ename string,dept string , salary double,hire_date date)").show()

# spark.sql("Select * from default.emp").show()
# spark.sql(""" 
#           insert into default.emp values \
#           (1,"ram","Sales",2500,'2010-01-01') \
#         , (2,"sam","Sales",2500,'2010-01-03') \
#         ,(3,"Mohan","IT",2000,'2015-03-01') \
#         , (4,"Mukesh","It",4500,'2015-03-05') \
#         , (5,"Manav","Biz",6500,'2015-05-15')
#           """).show()

spark.sql("Select * from default.emp").show()

+---+------+-----+------+----------+
|eid| ename| dept|salary| hire_date|
+---+------+-----+------+----------+
|  1|   ram|Sales|2500.0|2010-01-01|
|  2|   sam|Sales|2500.0|2010-01-03|
|  3| Mohan|   IT|2000.0|2015-03-01|
|  4|Mukesh|   It|4500.0|2015-03-05|
|  5| Manav|  Biz|6500.0|2015-05-15|
+---+------+-----+------+----------+



In [15]:
# save a delta table as table
try:
    df=spark.read.table("samples.nyctaxi.trips")
    df.write.mode("overwrite").saveAsTable("default.nyc_taxi")
except Exception as e:
    print(e)
# df.show(10)

In [16]:
spark.sql("select count(1) from default.nyc_taxi").show()

+--------+
|count(1)|
+--------+
|   21932|
+--------+



In [23]:
spark.sql(""" 
          Create table if not exists default.premium_trips as 
          select * from default.nyc_taxi
          where cast (fare_amount as double) >10
          """).show()
# spark.sql("select * from default.nyc_taxi order by fare_amount desc ")

+-----------------+-----------------+
|num_affected_rows|num_inserted_rows|
+-----------------+-----------------+
+-----------------+-----------------+



In [26]:
# spark.table("default.nyc_taxi").printSchema()
spark.sql("select * from default.premium_trips").show()

+--------------------+---------------------+-------------+-----------+----------+-----------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|fare_amount|pickup_zip|dropoff_zip|
+--------------------+---------------------+-------------+-----------+----------+-----------+
| 2016-01-04 18:58:23|  2016-01-04 18:58:45|          0.0|      105.0|      7114|       7114|
| 2016-01-28 17:36:17|  2016-01-28 18:27:48|         21.3|      115.0|     10017|      11042|
| 2016-01-16 18:09:15|  2016-01-16 18:09:23|          0.0|      105.0|      7310|       7310|
| 2016-02-17 22:23:14|  2016-02-17 23:06:15|        25.46|      130.0|     10103|       7901|
| 2016-02-29 12:16:16|  2016-02-29 12:16:53|          0.0|      260.0|      8876|       8876|
| 2016-02-12 20:55:19|  2016-02-12 21:52:38|        20.85|      275.0|     10013|       7008|
| 2016-01-30 22:28:42|  2016-01-30 22:30:14|          0.0|      188.0|      7974|       7974|
+--------------------+---------------------+-------------+--

In [9]:
# running using .venv_dvx --20260517
#Insertion
# spark.sql("select * from default.emp").show()
# spark.sql(""" 
# Insert into default.emp values \
#           (6,"Rahul","IT",8800,"2018-05-13") \
#           ,(7,"Santsh","Biz",8000,"2018-04-13")
#           """)

# spark.sql("select * from default.emp").show()

#Update
# spark.sql("""
#           update default.emp \
#           set salary=salary*1.1 \
#           where upper(dept)="IT"
#           """).show()
spark.sql("select * from default.emp").show()

+---+------+-----+------+----------+
|eid| ename| dept|salary| hire_date|
+---+------+-----+------+----------+
|  1|   ram|Sales|2500.0|2010-01-01|
|  2|   sam|Sales|2500.0|2010-01-03|
|  5| Manav|  Biz|6500.0|2015-05-15|
|  3| Mohan|   IT|2200.0|2015-03-01|
|  4|Mukesh|   It|4950.0|2015-03-05|
|  6| Rahul|   IT|9680.0|2018-05-13|
|  7|Santsh|  Biz|8000.0|2018-04-13|
+---+------+-----+------+----------+



In [12]:
#Update
spark.sql("""
          update default.emp \
          set salary=salary*1.1 \
          where upper(dept)="SALES"
          """).show()
spark.sql("select * from default.emp").show()

+-----------------+
|num_affected_rows|
+-----------------+
|                2|
+-----------------+



+---+------+-----+------+----------+
|eid| ename| dept|salary| hire_date|
+---+------+-----+------+----------+
|  5| Manav|  Biz|6500.0|2015-05-15|
|  3| Mohan|   IT|2200.0|2015-03-01|
|  4|Mukesh|   It|4950.0|2015-03-05|
|  6| Rahul|   IT|9680.0|2018-05-13|
|  7|Santsh|  Biz|8000.0|2018-04-13|
|  1|   ram|Sales|2750.0|2010-01-01|
|  2|   sam|Sales|2750.0|2010-01-03|
+---+------+-----+------+----------+



In [14]:
#Deletion
spark.sql("""select * from default.emp
          """).show()
spark.sql("""
          Delete from default.emp where eid="7"
          """).show()          

+---+------+-----+------+----------+
|eid| ename| dept|salary| hire_date|
+---+------+-----+------+----------+
|  1|   ram|Sales|2750.0|2010-01-01|
|  2|   sam|Sales|2750.0|2010-01-03|
|  5| Manav|  Biz|6500.0|2015-05-15|
|  3| Mohan|   IT|2200.0|2015-03-01|
|  4|Mukesh|   It|4950.0|2015-03-05|
|  6| Rahul|   IT|9680.0|2018-05-13|
|  7|Santsh|  Biz|8000.0|2018-04-13|
+---+------+-----+------+----------+



+-----------------+
|num_affected_rows|
+-----------------+
|                1|
+-----------------+



In [15]:
spark.sql("""select * from default.emp
          """).show()

+---+------+-----+------+----------+
|eid| ename| dept|salary| hire_date|
+---+------+-----+------+----------+
|  1|   ram|Sales|2750.0|2010-01-01|
|  2|   sam|Sales|2750.0|2010-01-03|
|  5| Manav|  Biz|6500.0|2015-05-15|
|  3| Mohan|   IT|2200.0|2015-03-01|
|  4|Mukesh|   It|4950.0|2015-03-05|
|  6| Rahul|   IT|9680.0|2018-05-13|
+---+------+-----+------+----------+



In [17]:
spark.sql("""
        describe history default.emp
""").show()

+-------+-------------------+----------------+--------------------+------------+--------------------+----+--------+-----------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|version|          timestamp|          userId|            userName|   operation| operationParameters| job|notebook|queryHistoryStatementId|           clusterId|readVersion|   isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+-------------------+----------------+--------------------+------------+--------------------+----+--------+-----------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|     10|2026-05-17 11:37:25|4266751462099821|radheybits@gmail.com|    OPTIMIZE|{clusterBy -> [],...|NULL|    NULL|   ac791420-c351-41b...|0517-112014-z3hfg...|          9|SnapshotIsolation|        false|{numRemovedFiles

In [24]:
spark.sql("""
            Select * from default.emp version as of 2 order by eid
          """)

,eid,ename,dept,salary,hire_date
0,1,ram,Sales,2500.0,2010-01-01
1,2,sam,Sales,2500.0,2010-01-03
2,3,Mohan,IT,2000.0,2015-03-01
3,4,Mukesh,It,4500.0,2015-03-05
4,5,Manav,Biz,6500.0,2015-05-15
5,6,Rahul,IT,8800.0,2018-05-13
6,7,Santsh,Biz,8000.0,2018-04-13


In [25]:
# RESTORE: RESTORE TABLE table_name TO VERSION AS OF <N> reverts a table to a previous version.